In [13]:
import pandas as pd

files = [
    'SCI-2025_v1.0_pathways_ensemble_R5_emissions.xlsx',
    'SCI-2025_v1.0_pathways_ensemble_R5_energy.xlsx',
    'SCI-2025_v1.0_pathways_ensemble_R5_other.xlsx'
]

df = pd.concat(
    [pd.read_excel(f, sheet_name='data') for f in files],
    ignore_index=True
)

print(df.shape)
print(df.head())

(1919649, 32)
         Model Scenario     Region              Variable       Unit      2010  \
0  AIM/CGE 2.0  SSP1-19  Asia (R5)          Emissions|BC   Mt BC/yr    2.9903   
1  AIM/CGE 2.0  SSP1-19  Asia (R5)         Emissions|CH4  Mt CH4/yr  150.0606   
2  AIM/CGE 2.0  SSP1-19  Asia (R5)   Emissions|CH4|AFOLU  Mt CH4/yr   79.5970   
3  AIM/CGE 2.0  SSP1-19  Asia (R5)  Emissions|CH4|Energy  Mt CH4/yr   44.0847   
4  AIM/CGE 2.0  SSP1-19  Asia (R5)          Emissions|CO   Mt CO/yr  317.9712   

        2015  2016  2017  2018  ...  2055      2060  2065      2070  2075  \
0    2.69235   NaN   NaN   NaN  ...   NaN    0.6121   NaN    0.5337   NaN   
1  156.82685   NaN   NaN   NaN  ...   NaN   22.2982   NaN   19.7144   NaN   
2   82.93980   NaN   NaN   NaN  ...   NaN   17.4418   NaN   15.8885   NaN   
3   44.62905   NaN   NaN   NaN  ...   NaN    3.1668   NaN    2.4041   NaN   
4  298.21915   NaN   NaN   NaN  ...   NaN  119.2289   NaN  107.1418   NaN   

      2080  2085     2090  2095     

In [14]:
df_asia = df[df['Region'] == 'Middle East & Africa (R5)']
df_asia = df_asia.drop(columns=['Region'])
print(df_asia.shape)
df_asia.head()

(378825, 31)


,Model,Scenario,Variable,Unit,2010,2015,2016,2017,2018,2019,...,2055,2060,2065,2070,2075,2080,2085,2090,2095,2100
44,AIM/CGE 2.0,SSP1-19,Emissions|BC,Mt BC/yr,1.8613,1.77580,NaN,NaN,NaN,NaN,...,NaN,1.3358,NaN,1.2784,NaN,1.2272,NaN,1.1624,NaN,1.1178
45,AIM/CGE 2.0,SSP1-19,Emissions|CH4,Mt CH4/yr,57.3106,62.94630,NaN,NaN,NaN,NaN,...,NaN,16.7021,NaN,16.7029,NaN,16.5006,NaN,16.7147,NaN,16.1650
46,AIM/CGE 2.0,SSP1-19,Emissions|CH4|AFOLU,Mt CH4/yr,26.7320,30.40350,NaN,NaN,NaN,NaN,...,NaN,13.6881,NaN,14.0738,NaN,14.1973,NaN,14.5880,NaN,14.2868
47,AIM/CGE 2.0,SSP1-19,Emissions|CH4|Energy,Mt CH4/yr,22.5380,23.15945,NaN,NaN,NaN,NaN,...,NaN,2.0853,NaN,1.7372,NaN,1.4485,NaN,1.2531,NaN,1.0489
48,AIM/CGE 2.0,SSP1-19,Emissions|CO,Mt CO/yr,241.8655,229.93740,NaN,NaN,NaN,NaN,...,NaN,177.9656,NaN,171.6242,NaN,165.9156,NaN,157.9494,NaN,152.8795


In [15]:
from openpyxl import load_workbook
from openpyxl.styles import Font, PatternFill, Alignment, Border, Side
from openpyxl.utils import get_column_letter


df.columns = df.columns.str.strip()

print("Columns found:", df_asia.columns.tolist())
print("Shape:", df_asia.shape)

Columns found: ['Model', 'Scenario', 'Variable', 'Unit', '2010', '2015', '2016', '2017', '2018', '2019', '2020', '2021', '2022', '2023', '2024', '2025', '2030', '2035', '2040', '2045', '2050', '2055', '2060', '2065', '2070', '2075', '2080', '2085', '2090', '2095', '2100']
Shape: (378825, 31)


In [16]:
df = df_asia

In [17]:
target_vars = [
    'Emissions|CO2',
    'Final Energy',
    'Primary Energy',
    'Primary Energy|Coal',
    'Primary Energy|Oil',
    'Primary Energy|Gas',
    'Primary Energy|Biomass',
    'Primary Energy|Nuclear',
    'Primary Energy|Solar',
    'Primary Energy|Wind',
    'Population',
    'Consumption',
    'GDP|PPP',
    ]

#target_vars = sorted(df['Variable'].dropna().unique())

# Auto-detect year columns (numeric column names like 2020, 2030, etc.)
year_cols = [c for c in df.columns if str(c).strip().isdigit()]
print("Year columns detected:", year_cols)

# Use last year if multiple, or whichever you prefer
# Change this to e.g. 2050 if you want a specific year
value_col = year_cols[-1] if year_cols else '2070'
print("Using year column:", value_col)

# ── Filter & Pivot ────────────────────────────────────────────────────────────
filtered = df[df['Variable'].isin(target_vars)].copy()

missing = set(target_vars) - set(filtered['Variable'].unique())
if missing:
    print("Warning: these variables not found in data:", missing)

pivot = filtered.pivot_table(
    index=['Model', 'Scenario'],
    columns='Variable',
    values=value_col,
    aggfunc='first'
).reset_index()

col_order = ['Model', 'Scenario'] + [v for v in target_vars if v in pivot.columns]
pivot = pivot[col_order]
units = filtered.groupby('Variable')['Unit'].first().to_dict()

print(f"Output shape: {pivot.shape}")

# ── Write & Format ─────────────────────────────────────────────────────────────
output_path = 'SCI_2025_reshaped_rows_asiaR5.xlsx'
pivot.to_excel(output_path, index=False, sheet_name='Data')

wb = load_workbook(output_path)
ws = wb['Data']

header_fill = PatternFill('solid', start_color='1F4E79', end_color='1F4E79')
header_font = Font(bold=True, color='FFFFFF', name='Arial', size=10)
center = Alignment(horizontal='center', vertical='center', wrap_text=True)
thin = Side(style='thin', color='CCCCCC')
border = Border(left=thin, right=thin, top=thin, bottom=thin)

for cell in ws[1]:
    cell.font = header_font
    cell.fill = header_fill
    cell.alignment = center
    cell.border = border

# Insert units row
ws.insert_rows(2)
unit_fill = PatternFill('solid', start_color='BDD7EE', end_color='BDD7EE')
unit_font = Font(italic=True, color='1F4E79', name='Arial', size=9)
ws['A2'] = 'Units:'
ws['A2'].font = Font(bold=True, italic=True, name='Arial', size=9, color='1F4E79')
ws['A2'].fill = unit_fill

for col_idx, col_name in enumerate(pivot.columns, 1):
    cell = ws.cell(row=2, column=col_idx)
    if col_name in units:
        cell.value = units[col_name]
    cell.font = unit_font
    cell.fill = unit_fill
    cell.alignment = center
    cell.border = border

alt_fill = PatternFill('solid', start_color='EBF3FB', end_color='EBF3FB')
white_fill = PatternFill('solid', start_color='FFFFFF', end_color='FFFFFF')
data_font = Font(name='Arial', size=10)

for row_idx, row in enumerate(ws.iter_rows(min_row=3, max_row=ws.max_row), 1):
    fill = alt_fill if row_idx % 2 == 0 else white_fill
    for col_idx, cell in enumerate(row, 1):
        cell.font = data_font
        cell.fill = fill
        cell.border = border
        if col_idx > 2 and cell.value is not None:
            cell.number_format = '#,##0.00'
            cell.alignment = Alignment(horizontal='right')
        else:
            cell.alignment = Alignment(horizontal='left')

ws.column_dimensions['A'].width = 22
ws.column_dimensions['B'].width = 14
for col_idx in range(3, len(pivot.columns) + 1):
    ws.column_dimensions[get_column_letter(col_idx)].width = 24

ws.freeze_panes = 'C3'
ws.row_dimensions[1].height = 40
wb.save(output_path)
print(f"Done → {output_path}")

Year columns detected: ['2010', '2015', '2016', '2017', '2018', '2019', '2020', '2021', '2022', '2023', '2024', '2025', '2030', '2035', '2040', '2045', '2050', '2055', '2060', '2065', '2070', '2075', '2080', '2085', '2090', '2095', '2100']
Using year column: 2100
Output shape: (1538, 15)
Done → SCI_2025_reshaped_rows_asiaR5.xlsx
